# Лабораторная 08. Data skew

Цель: увидеть, как один слишком частый ключ делает одну task намного тяжелее остальных.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.appName('lab-08-skew').master('local[*]')
    .config('spark.driver.memory', '2g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.adaptive.enabled', 'false')
    .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
print('Spark UI:', spark.sparkContext.uiWebUrl)

## Создаём перекошенный DataFrame
80% строк имеют `key = 'hot_key'`, остальные распределены по многим ключам.

In [ ]:
n = 400_000
skewed = (
    spark.range(n)
    .withColumn('key', F.when(F.col('id') < n * 0.8, F.lit('hot_key')).otherwise(F.concat(F.lit('key_'), (F.col('id') % 1000))))
    .withColumn('value', F.rand(7))
    .repartition(8)
)
skewed.groupBy('key').count().orderBy(F.desc('count')).show(10)

## groupBy по skewed key
Запустите aggregation и откройте Spark UI -> Stages. Ищите task duration: одна task может быть заметно дольше.

In [ ]:
result = skewed.groupBy('key').agg(F.count('*').alias('cnt'), F.sum('value').alias('sum_value'))
result.explain('formatted')
result.count()

## Skewed join
Join по перекошенному ключу часто ещё заметнее, потому что тяжёлая reduce partition должна сопоставить много строк.

In [ ]:
dim = spark.createDataFrame([(f'key_{i}', f'name_{i}') for i in range(1000)] + [('hot_key', 'very_hot')], ['key', 'name'])
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')
joined = skewed.join(dim, 'key')
joined.explain('formatted')
joined.count()

Вопросы:

- Почему один key создаёт проблему?
- Почему один task работает дольше?
- Что видно в Task Duration и Shuffle Read?
- Почему простое увеличение памяти не всегда решает skew?
- Какие способы борьбы возможны: AQE skew join, salting, изменение ключа, предварительная агрегация, broadcast маленькой стороны?